In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set academic plotting style for paper
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'serif',
    'axes.labelsize': 14,
    'legend.fontsize': 12,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
})

def main():
    # 1. Load Data
    try:
        data = np.load('simulation_data.npz')
    except FileNotFoundError:
        print("Error: 'simulation_data.npz' not found. Run the generation script first.")
        return

    # Original simulation parameters (m_sim = 2 to 26)
    ms_sim = data['ms']
    fast_times_all = data['fast_times']       # shape: (len(ms), num_trials)

    # 2. Map to True Degree (4d = 2^{m_sim} => 2^{m+2} = 2^{m_sim} => m = m_sim - 2)
    actual_ms = ms_sim - 2

    # Filter to only include m=1 to m=24 as requested
    mask = (actual_ms >= 1) & (actual_ms <= 24)

    ms = actual_ms[mask]
    degrees = 2.0**ms  # True degree d = 2^m
    fast_times = fast_times_all[mask]

    # 3. Setup Single Figure
    fig, ax = plt.subplots(figsize=(7, 5))

    # Calculate statistics for Fast method: Mean, Min, and Max
    fast_means = np.mean(fast_times, axis=1)
    fast_mins = np.min(fast_times, axis=1)
    fast_maxs = np.max(fast_times, axis=1)

    # Matplotlib error bars require relative offsets from the mean
    y_err = [fast_means - fast_mins, fast_maxs - fast_means]

    # Plot empirical mean with min/max error bars
    ax.errorbar(degrees, fast_means, yerr=y_err, fmt='-o', color='#1f77b4',
                capsize=4, capthick=1.5, ecolor='#1f77b4', label='EQSP (Algorithm 1)')

    # Theoretical O(d log d) anchor with a visual offset
    anchor_idx = -1
    visual_offset = 5.0 # Multiplier to shift the line upwards (adjust as needed)

    # Calculate base constant, then multiply by offset
    scaling_const = (fast_means[anchor_idx] * visual_offset) / (degrees[anchor_idx] * np.log2(degrees[anchor_idx]))
    theoretical = scaling_const * degrees * np.log2(degrees)

    ax.plot(degrees, theoretical, 'k--', label=r'$\mathcal{O}(d \log d)$')

    # Formatting
    ax.set_xscale('log', base=2)
    ax.set_yscale('log', base=10)
    ax.set_xlabel('Degree $d = 2^m$')
    ax.set_ylabel('Execution Time (s)')
    ax.grid(True, which='major', linestyle='-', alpha=0.5)
    ax.grid(True, which='minor', linestyle=':', alpha=0.2)
    ax.legend()

    plt.tight_layout()
    plt.savefig('qsp_numerical_scaling.pdf', dpi=300, bbox_inches='tight')
    print("[SUCCESS] Plot saved as 'qsp_numerical_scaling.pdf'")
    plt.show()

if __name__ == '__main__':
    main()